# 02_01_正解率_ベースライン_リーク

コード：`ch02/blocks/02_01_正解率_ベースライン_リーク.ipynb`

本文のコードをコードセルで直接実行します。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'scikit-learn>=1.5,<2']], check=True)


In [ ]:
# 2 章：正解率の落とし穴・ベースラインとの比較・データのリークを、scikit-learn で確かめる
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, average_precision_score

# ① 陽性が 1% のデータで「全部陰性」と答える
X, y = make_classification(n_samples=20000, n_features=20, n_informative=6,
                           weights=[0.99], flip_y=0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
always_neg = np.zeros_like(yte)
print(f"① 陽性の割合 {yte.mean():.3f}  全部陰性：正解率 {accuracy_score(yte, always_neg):.3f}  "
      f"再現率 {recall_score(yte, always_neg):.3f}")

# ② ロジスティック回帰（ベースライン）と勾配ブースティング
for name, model in [("ロジスティック回帰", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
                    ("勾配ブースティング", HistGradientBoostingClassifier(random_state=0))]:
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    pred = (p >= 0.5).astype(int)
    print(f"② {name:10s} 正解率 {accuracy_score(yte, pred):.3f}  適合率 {precision_score(yte, pred):.3f}  "
          f"再現率 {recall_score(yte, pred):.3f}  F1 {f1_score(yte, pred):.3f}  PR-AUC {average_precision_score(yte, p):.3f}")
    # 閾値を下げると再現率と適合率が入れ替わる
    pred2 = (p >= 0.2).astype(int)
    print(f"   閾値 0.2：適合率 {precision_score(yte, pred2):.3f}  再現率 {recall_score(yte, pred2):.3f}")

# ③ リーク：ラベルと無関係な乱数の特徴量から、分割の前に「効く 20 個」を選ぶ
rng = np.random.default_rng(0)
Xn, yn = rng.normal(size=(200, 5000)), rng.integers(0, 2, 200)      # 本当は何も学べない
leaky = SelectKBest(f_classif, k=20).fit(Xn, yn).transform(Xn)      # 全データで選んでしまう
acc_leak = cross_val_score(LogisticRegression(max_iter=1000), leaky, yn, cv=5).mean()
honest = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
acc_ok = cross_val_score(honest, Xn, yn, cv=5).mean()                # 各分割の訓練側だけで選ぶ
print(f"③ 乱数の特徴量：分割前に選ぶと正解率 {acc_leak:.3f}  分割の中で選ぶと {acc_ok:.3f}（偶然は 0.5）")
